In [9]:
import sys
from pathlib import Path

# go to project root (notebooks -> parent)
project_root = Path.cwd().parent
sys.path.append(str(project_root))

In [15]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.impute import SimpleImputer
from src import load_raw_data


In [16]:
df = load_raw_data()
X= df.drop('loan_status', axis=1)
y= df['loan_status']

In [18]:
num_cols = X.select_dtypes(include=['int64','float64']).columns
cat_cols = X.select_dtypes(include=['object']).columns

# ---- Preprocessing Pipelines ----
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocess = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_cols),
        ('cat', categorical_transformer, cat_cols)
    ]
)

# ---- Final Pipeline ----
model = Pipeline(steps=[
    ('preprocess', preprocess),
    ('clf', LogisticRegression(max_iter=1000))
])

# ---- Train-Test Split ----
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    stratify=y, 
    random_state=42
)

# ---- Train ----
model.fit(X_train, y_train)

# ---- Predict ----
pred_proba = model.predict_proba(X_test)[:, 1]

# ---- AUC ----
auc = roc_auc_score(y_test, pred_proba)
print("Baseline AUC:", auc)


Baseline AUC: 0.8693385451388458
